## 필수 · 기본 문제 1. 워크플로우 누락·순서 검증기

### 문제 배경

체크리스트에 단계 이름이 있어도 평가가 빠졌거나 split과 tokenization 순서가 바뀌면 재현성과 데이터 누수 관리가 약해집니다. 필수 단계의 존재·알 수 없는 단계·순서를 함께 검사합니다.

### 시작 코드

```python
REQUIRED = ["problem_definition", "data_split", "tokenize", "model", "train", "evaluate", "save"]
normal = REQUIRED.copy()
broken = ["problem_definition", "tokenize", "data_split", "model", "train", "save"]

def validate_workflow(stages):
    raise NotImplementedError
```

### 수행 요구사항

1. `missing`, `unknown`, `duplicates`, `order_ok`, `valid`를 반환하세요.
2. 필수 단계가 한 번씩 있고, unknown이 없고, 기준 순서일 때만 valid여야 합니다.
3. 정상과 오류 목록을 모두 검증하세요.

### 제출 결과

- 두 workflow report
- 오류 workflow가 실패한 구체적 이유
- `기본 문제 1 자동 검증: PASS`

### 자동 검증

```python
assert validate_workflow(normal)["valid"] is True
report = validate_workflow(broken)
assert report["missing"] == ["evaluate"] and report["order_ok"] is False
print("기본 문제 1 자동 검증: PASS")
```
    

    
    **자주 하는 실수**
    
    - `set(stages) == set(REQUIRED)`만 검사합니다.
    - 필수 단계가 두 번 있어도 valid로 처리합니다.
    - `evaluate`와 inference를 같은 단계로 취급합니다.

In [2]:
from collections import Counter

REQUIRED = ["problem_definition", "data_split", "tokenize", "model", "train", "evaluate", "save"]
normal = REQUIRED.copy()
broken = ["problem_definition", "tokenize", "data_split", "model", "train", "save"]

def validate_workflow(stages):
    # Counter 하나로 누락과 중복을 함께 계산합니다.
    counts = Counter(stages)
    missing = [stage for stage in REQUIRED if counts[stage] == 0]
    unknown = [stage for stage in stages if stage not in REQUIRED]
    duplicates = sorted(stage for stage, count in counts.items() if count > 1)
    # 존재하는 필수 단계만 기준 순서로 투영해 순서 뒤바뀜을 찾습니다.
    projected = [stage for stage in stages if stage in REQUIRED]
    expected_present = [stage for stage in REQUIRED if stage in projected]
    order_ok = projected == expected_present
    valid = not missing and not unknown and not duplicates and order_ok
    return {"missing": missing, "unknown": unknown, "duplicates": duplicates,
            "order_ok": order_ok, "valid": valid}

print("normal:", validate_workflow(normal))
print("broken:", validate_workflow(broken))
assert validate_workflow(normal)["valid"] is True
report = validate_workflow(broken)
assert report["missing"] == ["evaluate"] and report["order_ok"] is False
print("기본 문제 1 자동 검증: PASS")

normal: {'missing': [], 'unknown': [], 'duplicates': [], 'order_ok': True, 'valid': True}
broken: {'missing': ['evaluate'], 'unknown': [], 'duplicates': [], 'order_ok': False, 'valid': False}
기본 문제 1 자동 검증: PASS


    **상세 해설** · Set 비교만 하면 중복과 순서를 잃습니다. 기준 목록에 따라 누락을 계산하고, 실제 목록을 필수 단계에 투영해 순서를 비교해야 합니다.

## 필수 · 기본 문제 2. Task별 실행 계약 생성기

### 문제 배경

분류와 생성은 tokenizer를 공유할 수 있지만 model head, logits, metric, 결과 해석이 다릅니다. Task spec에서 완전한 contract를 생성합니다.

### 시작 코드

```python
task_specs = [
    {"name": "뉴스 분류", "task": "classification", "num_labels": 3},
    {"name": "답변 생성", "task": "generation", "max_new_tokens": 64},
]

def build_task_plan(spec):
    raise NotImplementedError
```

### 수행 요구사항

1. 분류는 `AutoModelForSequenceClassification`, `[B,C]`, accuracy/macro-F1을 사용하세요.
2. 생성은 `AutoModelForCausalLM`, `[B,L,V]`, 품질 검토/latency를 사용하세요.
3. 각 plan에 tokenizer, loss, output, completion check를 포함하세요.
4. 지원하지 않는 task는 `ValueError`를 내세요.

### 제출 결과

- 두 task plan
- 같은 tokenizer API로 model head를 대체할 수 없는 이유
- `기본 문제 2 자동 검증: PASS`

### 자동 검증

```python
plans = [build_task_plan(spec) for spec in task_specs]
assert plans[0]["logits_shape"] == "[B, C]"
assert plans[1]["output"] == "generated_token_ids"
print("기본 문제 2 자동 검증: PASS")
```

**자주 하는 실수**

- Base encoder를 분류 head가 학습된 모델처럼 사용합니다.
- 생성 logits에 문장 분류 argmax를 적용합니다.
- 생성 품질을 loss 하나로만 평가합니다.

In [3]:
task_specs = [
    {"name": "뉴스 분류", "task": "classification", "num_labels": 3},
    {"name": "답변 생성", "task": "generation", "max_new_tokens": 64},
]

def build_task_plan(spec):
    # 분류는 한 샘플당 C개 점수를 내므로 [B,C] 계약을 사용합니다.
    if spec["task"] == "classification":
        if spec.get("num_labels", 0) < 2:
            raise ValueError("classification에는 num_labels>=2가 필요합니다.")
        return {
            "tokenizer": "AutoTokenizer",
            "model": "AutoModelForSequenceClassification",
            "loss": "cross_entropy",
            "logits_shape": "[B, C]",
            "metrics": ["accuracy", "macro_f1"],
            "output": "label_id",
            "completion_check": "validation으로 선택 후 test 1회",
        }
    # 생성은 위치마다 vocabulary 점수를 내고 별도 반복 decoding이 필요합니다.
    if spec["task"] == "generation":
        if spec.get("max_new_tokens", 0) < 1:
            raise ValueError("generation에는 max_new_tokens>=1이 필요합니다.")
        return {
            "tokenizer": "AutoTokenizer",
            "model": "AutoModelForCausalLM",
            "loss": "causal_lm",
            "logits_shape": "[B, L, V]",
            "metrics": ["quality_review", "latency"],
            "output": "generated_token_ids",
            "completion_check": "prompt 제거 후 새 token만 decode·검토",
        }
    raise ValueError(f"지원하지 않는 task: {spec['task']}")

plans = [build_task_plan(spec) for spec in task_specs]
for spec, plan in zip(task_specs, plans):
    print(spec["name"], "->", plan)
assert plans[0]["logits_shape"] == "[B, C]"
assert plans[1]["output"] == "generated_token_ids"
print("기본 문제 2 자동 검증: PASS")

뉴스 분류 -> {'tokenizer': 'AutoTokenizer', 'model': 'AutoModelForSequenceClassification', 'loss': 'cross_entropy', 'logits_shape': '[B, C]', 'metrics': ['accuracy', 'macro_f1'], 'output': 'label_id', 'completion_check': 'validation으로 선택 후 test 1회'}
답변 생성 -> {'tokenizer': 'AutoTokenizer', 'model': 'AutoModelForCausalLM', 'loss': 'causal_lm', 'logits_shape': '[B, L, V]', 'metrics': ['quality_review', 'latency'], 'output': 'generated_token_ids', 'completion_check': 'prompt 제거 후 새 token만 decode·검토'}
기본 문제 2 자동 검증: PASS


설계 포인트 · 분류와 생성이 공유하는 필드는 tokenizer뿐입니다. Model class, logits shape, metric, 후처리 계약을 task 분기 안에서 한 번에 반환해 누락을 막습니다.

상세 해설 · Tokenizer는 문자열을 ID로 바꾸는 공통 입구입니다. 분류 head는 문장별 클래스 score를, Causal LM head는 위치별 vocabulary score를 내므로 loss·후처리·평가까지 함께 달라집니다.